# Run the kidney segmentation model on your own scan

> **Research demo, not for clinical use.** Use anonymized scans only. The model makes real errors,
> and it is weakest on small tumors and on cysts.

This notebook runs on your own free Kaggle or Colab GPU, so your scan stays in your session.

1. Turn on a **GPU** and **Internet** in the notebook settings.
2. Add your `.nii` or `.nii.gz` CT scan. On Kaggle, use *Add Input*. On Colab, upload it to the
   Files panel. If you set no path below, the notebook uses the first scan it finds.
3. Run all the cells. It takes about 8 minutes per scan on a Kaggle GPU, plus a few minutes of setup.

The weights are derived from KiTS23 and are licensed CC BY-NC-SA 4.0: non-commercial use only.

The code cells are generated by `scripts/notebooks/build_run_notebook.py`.
Edit the repository, not this notebook.

In [ ]:
# Step 0 - settings
MODEL_REPO = ""  # the Hugging Face model repository, for example "your-name/kits23-nnunet"
SCAN = ""        # path to your scan; leave empty to use the first one found

In [ ]:
# Step 1 - folders and packages
import os, subprocess, sys
from pathlib import Path
WORK = "/kaggle/working/project23d"
for sub in ['src/project3d', 'src/project3d/data', 'src/project3d/eval', 'src/project3d/inference']:
    os.makedirs(f"{WORK}/{sub}", exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nnunetv2==2.8.1", "nibabel"], check=True)
for var in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"):  # silences nnU-Net warnings
    os.environ.setdefault(var, f"/tmp/{var}")

Write the project code (generated from the repository).

In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/__init__.py
"""Project_23D: renal CT 3D segmentation."""


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/data/__init__.py
"""KiTS23 data pipeline (SPEC 002): verify, features, split, nnU-Net export."""


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/data/features.py
"""Per-case label features used to stratify the split (D-012).

Only segmentations are read. Labels: 0 background, 1 kidney, 2 tumor, 3 cyst.
"""

from __future__ import annotations

from pathlib import Path

import numpy as np

VALID_LABELS = {0, 1, 2, 3}
KITS21_LAST_ID = 299  # cases 0-299 are the KiTS21 cohort; 400+ were added in KiTS23


def cohort_of(case_id: str) -> str:
    """Cohort proxy for contrast phase (D-010, D-012); not a phase label."""
    return "kits21" if int(case_id.split("_")[1]) <= KITS21_LAST_ID else "kits23_new"


def label_features(seg: np.ndarray, spacing_mm: tuple[float, ...]) -> dict:
    """Volumes in millilitres and presence flags from a label array."""
    present = {int(v) for v in np.unique(seg)}
    unknown = present - VALID_LABELS
    if unknown:
        raise ValueError(f"unexpected label values {sorted(unknown)}")
    counts = np.bincount(seg.ravel().astype(np.int64), minlength=4)
    voxel_ml = float(np.prod(spacing_mm)) / 1000.0
    return {
        "kidney_ml": round(counts[1] * voxel_ml, 3),
        "tumor_ml": round(counts[2] * voxel_ml, 3),
        "cyst_ml": round(counts[3] * voxel_ml, 3),
        "has_tumor": bool(counts[2] > 0),
        "has_cyst": bool(counts[3] > 0),
    }


def case_features(case_id: str, seg_path: str | Path) -> dict:
    """Read one segmentation and return its features plus header geometry."""
    import nibabel as nib

    img = nib.load(str(seg_path))
    seg = np.asanyarray(img.dataobj)
    if not np.all(np.mod(seg, 1) == 0):
        raise ValueError(f"{case_id}: non-integer label values")
    seg = seg.astype(np.uint8)
    spacing = tuple(float(z) for z in img.header.get_zooms()[:3])
    row = {"case_id": case_id, "cohort": cohort_of(case_id),
           "shape": "x".join(map(str, seg.shape)),
           "spacing_mm": "x".join(f"{s:.4f}" for s in spacing)}
    row.update(label_features(seg, spacing))
    return row


def geometry_mismatch(image_path: str | Path, seg_path: str | Path, atol: float = 1e-3) -> str | None:
    """Compare image and mask headers (no voxel data read). None if they match."""
    import nibabel as nib

    img, seg = nib.load(str(image_path)), nib.load(str(seg_path))
    if img.shape[:3] != seg.shape[:3]:
        return f"shape {img.shape[:3]} vs {seg.shape[:3]}"
    if not np.allclose(img.affine, seg.affine, atol=atol):
        return "affine differs"
    return None


def add_size_bins(rows: list[dict]) -> tuple[float, float]:
    """Add ``size_bin`` in place: tertiles of tumor volume among tumor cases.

    Cases without tumor get ``none``. Returns the two cut points (ml) so they
    can be recorded with the split.
    """
    vols = np.array([r["tumor_ml"] for r in rows if r["has_tumor"]], dtype=float)
    if vols.size == 0:
        cuts = (0.0, 0.0)
    else:
        q1, q2 = np.quantile(vols, [1 / 3, 2 / 3])
        cuts = (float(q1), float(q2))
    for r in rows:
        if not r["has_tumor"]:
            r["size_bin"] = "none"
        elif r["tumor_ml"] <= cuts[0]:
            r["size_bin"] = "small"
        elif r["tumor_ml"] <= cuts[1]:
            r["size_bin"] = "medium"
        else:
            r["size_bin"] = "large"
    return cuts


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/eval/__init__.py
"""SPEC 004-005: test-set metrics, statistics and case analysis."""


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/eval/postprocess.py
"""Post-processing chosen on the fold-0 validation cases (D-021)."""

from __future__ import annotations

import numpy as np
from scipy import ndimage

N_KIDNEYS = 2
CONNECTIVITY = np.ones((3, 3, 3), dtype=bool)  # 26-connected


def keep_largest_regions(seg: np.ndarray, n: int = N_KIDNEYS) -> np.ndarray:
    """Clear everything except the ``n`` largest connected foreground regions.

    Labels inside the kept regions do not change. A label map with ``n`` regions or
    fewer is returned as is.
    """
    regions, count = ndimage.label(seg > 0, structure=CONNECTIVITY)
    if count <= n:
        return seg
    sizes = np.bincount(regions.ravel())
    sizes[0] = 0  # background
    keep = np.argsort(sizes)[-n:]
    return np.where(np.isin(regions, keep), seg, 0).astype(seg.dtype)


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/inference/__init__.py
"""Inference used by the command line, the benchmark and the live service."""


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/inference/predictor.py
"""SPEC 008: the predictor used by the command line, the benchmark and the live service.

``Predictor`` holds everything that must stay true for every caller: input checks, the
post-processing rule from evaluation (D-021), the geometry check and the region volumes.
``NnUNetModel`` is the only part that needs nnU-Net and a GPU, so the rest is tested locally
with a fake model.
"""

from __future__ import annotations

import shutil
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Protocol

import numpy as np

from project3d.data import features
from project3d.eval.postprocess import keep_largest_regions

CHECKPOINT = "checkpoint_final.pth"
TRAINER_FILE = "project23d_trainer.py"  # the name it gets inside nnU-Net's package
LABELS = {"kidney": 1, "tumor": 2, "cyst": 3}
MAX_VOXELS = 300_000_000  # about 512 x 512 x 1150, larger than any scan used in training
SPACING_RANGE_MM = (0.1, 20.0)


class InvalidInput(ValueError):
    """The scan can't be used. The message is written for the person who supplied it."""


@dataclass(frozen=True)
class Options:
    """Inference settings. The defaults are the setting used in the evaluation (SPEC 004)."""

    tta: bool = True
    tile_step: float = 0.5
    device: str = "cuda"

    def __post_init__(self) -> None:
        if not 0 < self.tile_step <= 1:
            raise ValueError(f"tile_step must be in (0, 1], got {self.tile_step}")


class Model(Protocol):
    def read(self, path: Path) -> tuple[np.ndarray, dict]: ...
    def predict(self, data: np.ndarray, properties: dict) -> np.ndarray: ...
    def write(self, seg: np.ndarray, path: Path, properties: dict) -> None: ...


def check_header(shape: tuple, spacing: tuple, max_voxels: int = MAX_VOXELS) -> None:
    """Raise InvalidInput for a scan that isn't 3D, has bad spacing or is too large."""
    if len(shape) != 3:
        raise InvalidInput(f"Expected a 3D scan, but this file has {len(shape)} dimensions {tuple(shape)}.")
    low, high = SPACING_RANGE_MM
    if not all(np.isfinite(s) and low <= s <= high for s in spacing[:3]):
        raise InvalidInput(f"Voxel spacing {tuple(float(s) for s in spacing[:3])} mm is invalid: "
                           f"each axis must be between {low} and {high} mm.")
    voxels = int(np.prod([int(n) for n in shape]))
    if voxels > max_voxels:
        raise InvalidInput(f"This scan has {voxels:,} voxels; the limit is {max_voxels:,}.")


def load_checked(path: str | Path, max_voxels: int = MAX_VOXELS):
    """Open a NIfTI file and check its header. Voxel data is not read."""
    import nibabel as nib

    try:
        img = nib.load(str(path))
    except Exception as exc:
        raise InvalidInput("The file could not be read as a NIfTI scan.") from exc
    check_header(img.shape, img.header.get_zooms(), max_voxels)
    return img


def region_volumes(seg: np.ndarray, spacing: tuple) -> dict[str, float]:
    """Volume in mL of each region. One mL is 1000 cubic millimetres."""
    voxel_ml = float(np.prod([float(s) for s in spacing[:3]])) / 1000
    return {f"{name}_ml": round(float((seg == label).sum()) * voxel_ml, 3) for name, label in LABELS.items()}


class Predictor:
    def __init__(self, model: Model, options: Options | None = None):
        self.model = model
        self.options = options or Options()

    def predict_file(self, src: str | Path, dst: str | Path) -> dict[str, Any]:
        """Segment one scan, write the labels to ``dst`` and return a summary.

        The summary has no file names, so a caller can log or store it.
        """
        src, dst = Path(src), Path(dst)
        img = load_checked(src)
        data, properties = self.model.read(src)
        seg = keep_largest_regions(np.asarray(self.model.predict(data, properties)).astype(np.uint8))
        self.model.write(seg, dst, properties)
        why = features.geometry_mismatch(dst, src)
        if why:
            raise RuntimeError(f"output does not match the input geometry ({why})")
        spacing = tuple(float(z) for z in img.header.get_zooms()[:3])
        return {"shape": [int(n) for n in img.shape], "spacing_mm": list(spacing),
                "voxel_count": int(np.prod([int(n) for n in img.shape])),
                "volumes_ml": region_volumes(seg, spacing), "options": asdict(self.options),
                "postprocessing": "keep the two largest foreground regions"}


def install_trainer(trainer_file: Path, variants_dir: Path | None = None) -> Path:
    """Copy our trainer into nnU-Net, which only looks for trainer classes in its own package.

    The checkpoint names that class, so loading fails without this step.
    """
    if variants_dir is None:
        import nnunetv2

        variants_dir = Path(nnunetv2.__file__).parent / "training" / "nnUNetTrainer" / "variants"
    return Path(shutil.copy(trainer_file, variants_dir / TRAINER_FILE))


class NnUNetModel:
    """The packaged nnU-Net model, reading files the same way training did."""

    def __init__(self, model_dir: str | Path, options: Options | None = None):
        options = options or Options()
        model_dir = Path(model_dir)
        install_trainer(model_dir / TRAINER_FILE)
        import torch
        from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

        self.predictor = nnUNetPredictor(tile_step_size=options.tile_step, use_mirroring=options.tta,
                                         device=torch.device(options.device), allow_tqdm=False)
        self.predictor.initialize_from_trained_model_folder(str(model_dir), use_folds=(0,),
                                                            checkpoint_name=CHECKPOINT)
        self.io = self.predictor.plans_manager.image_reader_writer_class()

    def read(self, path: Path) -> tuple[np.ndarray, dict]:
        return self.io.read_images([str(path)])

    def predict(self, data: np.ndarray, properties: dict) -> np.ndarray:
        # runs in this process: nnU-Net's worker processes can't be used inside a ZeroGPU call
        return self.predictor.predict_single_npy_array(data, properties)

    def write(self, seg: np.ndarray, path: Path, properties: dict) -> None:
        self.io.write_seg(seg, str(path), properties)


In [ ]:
# Step 2 - find the model and the scan
import torch
from huggingface_hub import snapshot_download
sys.path.insert(0, f"{WORK}/src")
ROOTS = [Path("/kaggle/input"), Path("/content")]
def search(pattern):
    return [p for root in ROOTS if root.exists() for p in sorted(root.rglob(pattern))]
packaged = search("checksums.json")
MODEL_DIR = packaged[0].parent if packaged else snapshot_download(MODEL_REPO)
scans = [p for suffix in ("*.nii.gz", "*.nii") for p in search(suffix)
         if p.name not in ("pred.nii.gz", "gt.nii.gz")]
assert SCAN or scans, "No scan found. Add a .nii or .nii.gz file, or set SCAN above."
SCAN = Path(SCAN or scans[0])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("scan:", SCAN.name, "| device:", DEVICE)

In [ ]:
# Step 3 - segment. Test-time augmentation stays on, as in the evaluation.
from project3d.inference.predictor import NnUNetModel, Options, Predictor
OPTIONS = Options(tta=True, device=DEVICE)
OUT = Path("segmentation.nii.gz").resolve()
summary = Predictor(NnUNetModel(MODEL_DIR, OPTIONS), OPTIONS).predict_file(SCAN, OUT)
print(summary["volumes_ml"])
print("Saved", OUT, "- labels: 1 kidney, 2 tumor, 3 cyst")

In [ ]:
# Step 4 - look at the slice with the most foreground. Research demo, not for clinical use.
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
ct = np.asanyarray(nib.load(str(SCAN)).dataobj)
seg = np.asanyarray(nib.load(str(OUT)).dataobj)
axis = int(np.argmin(seg.shape))
others = tuple(i for i in range(3) if i != axis)
index = int((seg > 0).sum(axis=others).argmax())
plt.figure(figsize=(6, 6))
plt.imshow(np.take(ct, index, axis).T, cmap="gray", vmin=-200, vmax=300, origin="lower")
plt.imshow(np.ma.masked_equal(np.take(seg, index, axis).T, 0), cmap="autumn", alpha=0.45, origin="lower")
plt.title(f"Slice {index}. Research demo, not for clinical use")
plt.axis("off")
plt.savefig("overlay.png", dpi=120, bbox_inches="tight")
plt.show()